# 11 — Well-Architected Framework: Presentation Walkthrough

> **What this notebook is.** A pillar-by-pillar map from the Databricks Well-Architected Framework to
> concrete cells + tables in the notebooks you just ran. Use it as your demo
> script with the MLB team: click into any row, show the cell, explain the
> trade-off, move on.

The seven pillars, in the order the MLB team will probably care about them:

1. **Data Governance**
2. **Interoperability & Usability**
3. **Operational Excellence**
4. **Security, Compliance & Privacy**
5. **Reliability**
6. **Performance Efficiency**
7. **Cost Optimization**

Run the cells below to produce a summary of everything this demo has deployed.


In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "mlb_gumbo_waf")
print(f"{UC_CATALOG}.{UC_SCHEMA}")


alexander_booth.mlb_gumbo_waf


## Pillar 1 — Data Governance

**Why WAF cares:** Data is useless (and often unsafe) without provenance,
ownership, and discoverability. UC gives you one catalog, one set of
permissions, and one lineage graph that covers everything.

**Where in this demo:**
- **Bronze / Silver / Gold schemas** with descriptive `COMMENT ON` → `00_verify_connection`
- **Table + column comments** that feed Catalog Explorer and Genie → `05_governance_tags_lineage`
- **Tags** (`mlb_analytics`, `tier`, `data_owner`) on every table so the asset shows up in the right Databricks One Domain → `05`
- **RELY PK/FK** constraints so the Catalog Explorer ERD draws the star schema correctly → `03`, `04`
- **Lineage** — no code required, `system.access.table_lineage` automatically shows bronze → silver → gold → ML → dashboard → `05`


In [2]:
print("Demo tag surface area:")
spark.sql(f"""
    SELECT tag_name, COUNT(*) AS tagged_tables
    FROM {UC_CATALOG}.information_schema.table_tags
    WHERE schema_name LIKE '{UC_SCHEMA}%'
    GROUP BY tag_name
    ORDER BY tagged_tables DESC
""").show(truncate=False)


Demo tag surface area:


+-------------+-------------+
|tag_name     |tagged_tables|
+-------------+-------------+
|tier         |7            |
|data_owner   |7            |
|mlb_analytics|7            |
+-------------+-------------+



## Pillar 2 — Interoperability & Usability

**Why WAF cares:** If the only way to use the data is writing Spark, you have
a data engineering team, not a data platform. WAF rewards shapes that are
both machine- and human-friendly.

**Where in this demo:**
- **Star schema** (`dim_*` + `fact_*`) — the shape AI/BI auto-detects → `04`
- **Pre-aggregated views** (`v_pitcher_leaderboard`, `v_pitch_mix_by_type`, `v_team_pitching_summary`) → `04`
- **Lakeview dashboard** with 6 pre-built panels, created 100% via API → `09`
- **Genie space** with table routing + baseball-specific instructions → `10`
- **Gold table on Iceberg** (optional extension): see the `MLB Unified Data Layer with Iceberg` folder for reading this data from Snowflake, Trino, etc.


## Pillar 3 — Operational Excellence

**Why WAF cares:** Humans-in-the-loop doesn't scale. WAF rewards pipelines that
are reproducible, observable, and operable without pagerduty.

**Where in this demo:**
- **Auto Loader `availableNow`** batch mode — idempotent, schedulable, recoverable from the checkpoint → `02`
- **Parameterized ingestion** (`MLB_SEASON`, `MLB_TEAM_ID`, date window) via `.env` — the same notebook runs for any season → `01`
- **MLflow run lineage** — every training run is logged with dataset, metrics, signature, tags → `08`
- **Champion/challenger alias** pattern — promotion is a metadata flip, not a deploy → `08`
- **DQ results persisted** (`dq_results` table) — freshness/pass-rate become a dashboard tile → `06`


## Pillar 4 — Security, Compliance & Privacy

**Why WAF cares:** Your customer's sensitive data in their platform, not yours.
WAF rewards "permissions as data" — if you can't query it, it's governed.

**Where in this demo:**
- **Column tagging for PII** (`contains_pii=y` on player names) → `05`
- **Column mask** on `fact_games.attendance` that returns `NULL` unless the caller is in `mlb_analytics_team` → `05`
- **System table lineage** + tag propagation means an audit query like "which tables contain PII and who queried them last week?" is a single `system.access.audit` join.
- **Network / workspace boundaries**: covered at the platform level — out of scope for this demo but worth calling out.


## Pillar 5 — Reliability

**Why WAF cares:** Silent data quality problems are the worst incidents — they
don't page, but they erode trust for weeks.

**Where in this demo:**
- **Auto Loader rescue-data column** — a source schema drift lands in `_rescued` instead of dropping → `02`
- **Surrogate keys** (`MD5(...)`) so silver is deterministically idempotent → `03`
- **`CHECK` constraints** enforcing physics (no 200mph pitches) → `03`
- **`RELY` foreign keys** wired through gold → `04`
- **Dedicated DQ expectations** for completeness / integrity / freshness / range with results persisted per run → `06`


In [3]:
print("Data-quality rule pass rate (last run):")
try:
    spark.sql(f"""
        WITH last_run AS (
            SELECT run_id FROM {UC_CATALOG}.{UC_SCHEMA}_gold.dq_results
            ORDER BY run_time DESC LIMIT 1
        )
        SELECT layer,
               COUNT(*)                                AS checks,
               SUM(CASE WHEN passed THEN 1 ELSE 0 END) AS passed,
               SUM(CASE WHEN NOT passed AND severity='error' THEN 1 ELSE 0 END) AS hard_failures
        FROM {UC_CATALOG}.{UC_SCHEMA}_gold.dq_results
        WHERE run_id IN (SELECT run_id FROM last_run)
        GROUP BY layer
        ORDER BY layer
    """).show(truncate=False)
except Exception as e:
    print(f"  (run notebook 06 first) — {str(e)[:120]}")


Data-quality rule pass rate (last run):


+------+------+------+-------------+
|layer |checks|passed|hard_failures|
+------+------+------+-------------+
|bronze|3     |2     |0            |
|gold  |3     |3     |0            |
|silver|3     |2     |0            |
+------+------+------+-------------+



## Pillar 6 — Performance Efficiency

**Why WAF cares:** Faster queries = happier users + lower DBU spend. Two
levers: lay data out to match the query pattern, and let the platform do
maintenance for you.

**Where in this demo:**
- **Liquid clustering** tuned to the Lakeview / Genie predicate shape:
  - `bronze.raw_gumbo` on `file_batch_time` (incremental silver loads)
  - `silver.game_data` on `(season, official_date)`
  - `silver.pitch_data` on `(season, official_date, game_pk)`
  - `gold.fact_pitches` on `(season, official_date, pitcher_sk)` — the hot table
- **Predictive Optimization** enabled on all three schemas — no more hand-rolled OPTIMIZE jobs → `07`
- **Serverless everywhere** — no cluster sizing debates, no idle-minute waste
- **Materialized views / pre-agg views** for the dashboard (`v_pitcher_leaderboard`, `v_pitch_mix_by_type`) → `04`


## Pillar 7 — Cost Optimization

**Why WAF cares:** You can only optimize what you can measure. WAF rewards
platforms where every DBU is attributable to a cost center, team, and use case.

**Where in this demo:**
- **Catalog / schema tags** (`cost_center`, `env`, `data_owner`, `removeAfter`) propagate to `system.billing.usage.custom_tags` → `00`
- **Tag-based chargeback query** against `system.billing.usage` → `07`
- **`removeAfter` stale-asset scanner** — a 5-line query that finds any tagged asset past its garbage-collection date → `07`
- **Serverless compute** — you don't pay for warm capacity you aren't using.
- **Automatic Predictive Optimization** — you stop writing OPTIMIZE jobs that run whether or not they were needed.


In [4]:
print("Chargeback surface — assets tagged with a removeAfter:")
spark.sql(f"""
    SELECT catalog_name, schema_name, table_name, tag_value AS remove_after
    FROM {UC_CATALOG}.information_schema.table_tags
    WHERE tag_name = 'removeAfter'
      AND schema_name LIKE '{UC_SCHEMA}%'
""").show(truncate=False)


Chargeback surface — assets tagged with a removeAfter:


+------------+-----------+----------+------------+
|catalog_name|schema_name|table_name|remove_after|
+------------+-----------+----------+------------+
+------------+-----------+----------+------------+



## Closing slide: what you'd say on stage

> *"Everything we ran today — the ingestion, the medallion pipeline, the data quality gates,
> the ML model, the dashboard, and the Genie space — all came from one catalog, with one
> set of governance tags, lineage, and audit records. That's what 'Well-Architected' looks
> like in practice on Databricks: pillars aren't posters on a wall, they're features of the
> platform that you can point at, cell by cell."*
